In [5]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import copy
from mininn.training import train_one_epoch, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet18_Weights.DEFAULT

In [2]:
class FrozenResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.backbone = resnet18(weights=weights)
        feature_count = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        for p in self.backbone.parameters():
            p.requires_grad_(False)

        self.classifier = nn.Linear(feature_count, num_classes)
        self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)

        # Keep BatchNorm statistics fixed, even during training.
        self.backbone.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)

        return self.classifier(features)


torch.manual_seed(42)
transfer_model = FrozenResNet18().to(device)

print("Classifier:", transfer_model.classifier)

transfer_model.train()
print("Backbone training:", transfer_model.backbone.training)
print("Classifier training:", transfer_model.classifier.training)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/devansh/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


Classifier: Linear(in_features=512, out_features=10, bias=True)
Backbone training: False
Classifier training: True


In [4]:
preprocess = weights.transforms()

transfer_dataset = datasets.CIFAR10(
    root="../data_1",
    train=True,
    download=True,
    transform=preprocess,
)

transfer_train_subset, transfer_val_subset = random_split(
    transfer_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42),
)

transfer_train_loader = DataLoader(
    transfer_train_subset,
    batch_size=64,
    shuffle=True,
)

transfer_train_eval_loader = DataLoader(
    transfer_train_subset,
    batch_size=64,
    shuffle=False,
)

transfer_val_loader = DataLoader(
    transfer_val_subset,
    batch_size=64,
    shuffle=False,
)

images, labels = next(iter(transfer_train_loader))
print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([64, 3, 224, 224])
Labels: torch.Size([64])


In [6]:
transfer_optimizer = torch.optim.AdamW(
    transfer_model.classifier.parameters(),
    lr=0.001,
    weight_decay=0.01,
)

best_accuracy = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, 6):
    train_loss = train_one_epoch(
        transfer_model,
        transfer_train_loader,
        transfer_optimizer,
        device,
    )

    val_metrics = evaluate(
        transfer_model, transfer_val_loader, device
    )

    if val_metrics["accuracy"] > best_accuracy:
        best_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = copy.deepcopy(transfer_model.state_dict())

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Val accuracy: {val_metrics['accuracy']:.2f}%"
    )

transfer_model.load_state_dict(best_state)

train_metrics = evaluate(
    transfer_model, transfer_train_eval_loader, device
)
val_metrics = evaluate(
    transfer_model, transfer_val_loader, device
)

print(f"\nRestored epoch: {best_epoch}")
print(f"Training accuracy: {train_metrics['accuracy']:.2f}%")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

torch.save(
    {
        "model_state": transfer_model.state_dict(),
        "epoch": best_epoch,
        "val_accuracy": val_metrics["accuracy"],
        "pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1",
    },
    "best_frozen_resnet18.pt",
)

Epoch 1 | Train loss: 0.6338 | Val loss: 0.4484 | Val accuracy: 84.58%
Epoch 2 | Train loss: 0.4286 | Val loss: 0.4081 | Val accuracy: 86.08%
Epoch 3 | Train loss: 0.3980 | Val loss: 0.3915 | Val accuracy: 86.58%
Epoch 4 | Train loss: 0.3837 | Val loss: 0.3908 | Val accuracy: 86.12%
Epoch 5 | Train loss: 0.3741 | Val loss: 0.3943 | Val accuracy: 85.92%

Restored epoch: 3
Training accuracy: 87.32%
Validation accuracy: 86.58%


In [7]:
class FineTuneResNet18(FrozenResNet18):
    def __init__(self, num_classes=10):
        super().__init__(num_classes)

        # Unfreeze only the final residual stage.
        for p in self.backbone.layer4.parameters():
            p.requires_grad_(True)

    def forward(self, x):
        # Gradient tracking is now enabled.
        features = self.backbone(x)
        return self.classifier(features)


finetune_model = FineTuneResNet18().to(device)
finetune_model.load_state_dict(transfer_model.state_dict())

finetune_optimizer = torch.optim.AdamW(
    [
        {
            "params": finetune_model.backbone.layer4.parameters(),
            "lr": 1e-5,
        },
        {
            "params": finetune_model.classifier.parameters(),
            "lr": 1e-4,
        },
    ],
    weight_decay=0.01,
)

In [8]:
baseline = evaluate(
    finetune_model, transfer_val_loader, device
)

best_accuracy = baseline["accuracy"]
best_epoch = 0  # Before fine-tuning
best_state = copy.deepcopy(finetune_model.state_dict())

for epoch in range(1, 6):
    train_loss = train_one_epoch(
        finetune_model,
        transfer_train_loader,
        finetune_optimizer,
        device,
    )

    val_metrics = evaluate(
        finetune_model, transfer_val_loader, device
    )

    if val_metrics["accuracy"] > best_accuracy:
        best_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = copy.deepcopy(finetune_model.state_dict())

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Val accuracy: {val_metrics['accuracy']:.2f}%"
    )

finetune_model.load_state_dict(best_state)

train_metrics = evaluate(
    finetune_model, transfer_train_eval_loader, device
)
val_metrics = evaluate(
    finetune_model, transfer_val_loader, device
)

print(f"\nRestored fine-tuning epoch: {best_epoch}")
print(f"Training accuracy: {train_metrics['accuracy']:.2f}%")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

torch.save(
    {
        "model_state": finetune_model.state_dict(),
        "finetune_epoch": best_epoch,
        "val_accuracy": val_metrics["accuracy"],
        "pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1",
    },
    "best_finetuned_resnet18.pt",
)

Epoch 1 | Train loss: 0.3273 | Val loss: 0.3051 | Val accuracy: 89.10%
Epoch 2 | Train loss: 0.2293 | Val loss: 0.2978 | Val accuracy: 89.66%
Epoch 3 | Train loss: 0.1712 | Val loss: 0.2697 | Val accuracy: 90.46%
Epoch 4 | Train loss: 0.1259 | Val loss: 0.2576 | Val accuracy: 91.36%
Epoch 5 | Train loss: 0.0909 | Val loss: 0.2646 | Val accuracy: 91.10%

Restored fine-tuning epoch: 4
Training accuracy: 97.79%
Validation accuracy: 91.36%


In [10]:
transfer_test_dataset = datasets.CIFAR10(
    root="../data_1",
    train=False,
    download=True,
    transform=preprocess,
)

transfer_test_loader = DataLoader(
    transfer_test_dataset,
    batch_size=64,
    shuffle=False,
)

test_metrics = evaluate(
    finetune_model,
    transfer_test_loader,
    device,
)

print(f"Test images: {len(transfer_test_dataset)}")
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.2f}%")

Test images: 10000
Test loss: 0.2755
Test accuracy: 91.05%
